# YouTube-bot Mobile Video Creator
Tap **Runtime → Run all**. Change the topic/mode in cell 2. At the end, download one ZIP containing the MP4 and all episode artifacts. If rendering fails, the notebook prints the exact FFmpeg error instead of hiding it.

In [ ]:
#@title 1) Install
!rm -rf YouTube-bot
!git clone -q https://github.com/studio1111/YouTube-bot.git
%cd YouTube-bot
!python -m pip install -q -e .
!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)
print('Ready. Run the next cell.')

In [ ]:
#@title 2) Create your video
TOPIC = 'Why the sky changes color' #@param {type:'string'}
EPISODE_ID = 'mobile-episode-001' #@param {type:'string'}
VIDEO_MODE = 'slides' #@param ['ken-burns', 'shorts-vertical', 'audiogram', 'openverse-broll', 'comfyui-workflow', 'slides']
!python -m yt_agent.cli run --topic "{TOPIC}" --episode-id "{EPISODE_ID}" --runtime-dir runtime --video-mode "{VIDEO_MODE}"
EPISODE_DIR = f'runtime/episodes/{EPISODE_ID}/v1'
print(f'Created: {EPISODE_DIR}')

In [ ]:
#@title 3) Diagnose and download
import shutil
from pathlib import Path
from google.colab import files
episode_dir = Path(EPISODE_DIR)
video = episode_dir / 'video.mp4'
if not video.exists():
    error = episode_dir / 'render_error.log'
    print(error.read_text() if error.exists() else 'No video and no render_error.log. Re-run cell 2 and inspect its output.')
    raise FileNotFoundError('MP4 was not rendered. The diagnostic is printed above.')
bundle = Path(f'{EPISODE_ID}-video-package')
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(episode_dir, bundle)
archive = shutil.make_archive(str(bundle), 'zip', root_dir=bundle)
files.download(archive)
print('Downloaded ZIP successfully.')